# 13. Ensemble of the four best models

Combines the exported predictions of the four base models
(**YOLO11-seg, DeepLabv3+ ResNet50, Mask2Former, U-Net ResNet50**) with three
averaging schemes:

1. **Simple average** - equal weights.
2. **Dice-weighted average** - weights proportional to the validation Dice of
   each model, normalized to sum to 1.
3. **Top-2 weighted** - only the two models with the highest validation Dice,
   with Dice-proportional weights.

The test split is kept out of every selection step:

1. The validation Dice of each base model is computed here from its exported
   validation predictions (per-scene hard Dice against the validation ground
   truth, the same definition for all four models); the ensemble weights are
   these scores normalized to sum to 1.
2. The three schemes are built and compared in both mask domains - **binary**
   exports and **probability** exports - on the **validation split**, and the
   scheme with the highest validation Dice is selected.
3. The selected scheme is evaluated once on the **test split**. Test scores of
   the non-selected schemes are printed for reference only and play no role in
   the selection.

Averaged maps are binarized at 0.5 (i.e. 127.5 on the 0-255 scale). The
notebook expects the validation and test exports of notebooks 02, 05, 10 and 12
(`val_masks/...`, `probability_masks/val/...`, `test_masks/...`,
`probability_masks/test/...`).

In [ ]:
import json
import os
from pathlib import Path

import cv2
import numpy as np

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Validation exports of the base models (produced by the validation-export
# cells of notebooks 02, 05, 10 and 12).
VAL_BINARY_MODEL_DIRS = {
    "YOLO": "val_masks/yolo",
    "dlv3+-ResNet50": "val_masks/dlv3+_with_resnet50",
    "Mask2Former": "val_masks/mask2former",
    "UNet-ResNet50": "val_masks/unet_with_resnet50",
}

VAL_PROBABILITY_MODEL_DIRS = {
    "YOLO": "probability_masks/val/yolo",
    "dlv3+-ResNet50": "probability_masks/val/dlv3+_with_resnet50",
    "Mask2Former": "probability_masks/val/mask2former",
    "UNet-ResNet50": "probability_masks/val/unet_with_resnet50",
}

# Test exports of the base models (produced by the test-export cells of the
# same notebooks).
TEST_BINARY_MODEL_DIRS = {
    "YOLO": "test_masks/yolo",
    "dlv3+-ResNet50": "test_masks/dlv3+_with_resnet50",
    "Mask2Former": "test_masks/mask2former",
    "UNet-ResNet50": "test_masks/unet_with_resnet50",
}

TEST_PROBABILITY_MODEL_DIRS = {
    "YOLO": "probability_masks/test/yolo",
    "dlv3+-ResNet50": "probability_masks/test/dlv3+_with_resnet50",
    "Mask2Former": "probability_masks/test/mask2former",
    "UNet-ResNet50": "probability_masks/test/unet_with_resnet50",
}

VAL_GT_PATH = "split_data/val/Masks"
TEST_GT_PATH = "split_data/test/Masks"

# Ensemble maps are written to <output dir>/<method>/.
OUTPUT_DIRS = {
    ("val", "binary"): "ensemble_val_binary",
    ("val", "probability"): "ensemble_val_probability",
    ("test", "binary"): "ensemble_test_binary",
    ("test", "probability"): "ensemble_test_probability",
}

MODEL_NAMES = list(VAL_BINARY_MODEL_DIRS.keys())

EVAL_THRESHOLD = 0.5   # averaged maps are binarized at threshold * 255

METHOD_TITLES = {
    "simple_average": "Simple average",
    "weighted_by_dice": "Dice-weighted average",
    "weighted_top2": "Top-2 by validation Dice",
}

## 2. Helpers

In [ ]:
import re


def match_filename(pred_name, gt_names):
    """Match a prediction file (<stem>_pred.png) to its ground-truth mask."""
    base_name = pred_name.replace("_pred.png", ".png")
    if base_name in gt_names:
        return base_name

    # Fall back to matching by the acquisition date in the file name.
    date_pattern = r"(\d{4}-\d{2}-\d{2})"
    match = re.search(date_pattern, pred_name)
    if match:
        date = match.group(1)
        for gt_name in gt_names:
            if date in gt_name:
                return gt_name

    if pred_name in gt_names:
        return pred_name

    return None


def calculate_dice(pred_mask, gt_mask, threshold=EVAL_THRESHOLD):
    """Dice score between a 0-255 prediction map and a 0-255 GT mask."""
    pred_binary = (pred_mask > threshold * 255).astype(np.uint8)
    gt_binary = (gt_mask > 127).astype(np.uint8)

    intersection = np.sum(pred_binary * gt_binary)
    union = np.sum(pred_binary) + np.sum(gt_binary)

    if union == 0:
        return 1.0
    return 2.0 * intersection / union


def evaluate_predictions_dir(pred_dir, gt_dir):
    """Mean/std Dice of all *_pred.png files in a directory against the GT."""
    gt_names = sorted(os.listdir(gt_dir))
    pred_files = sorted([p.name for p in Path(pred_dir).glob("*_pred.png")])

    dice_scores = []
    for pred_name in pred_files:
        gt_name = match_filename(pred_name, gt_names)
        if gt_name is None:
            continue

        pred = cv2.imread(str(Path(pred_dir) / pred_name), cv2.IMREAD_GRAYSCALE)
        gt = cv2.imread(str(Path(gt_dir) / gt_name), cv2.IMREAD_GRAYSCALE)
        if pred is None or gt is None:
            continue

        if pred.shape != gt.shape:
            pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]),
                              interpolation=cv2.INTER_NEAREST)

        dice_scores.append(calculate_dice(pred, gt))

    if not dice_scores:
        return None, None
    return float(np.mean(dice_scores)), float(np.std(dice_scores))

## 3. Validation Dice of the base models and ensemble weights

Per-scene hard Dice of the exported binary validation masks against the
validation ground truth, averaged over scenes - the same definition for all
four models. These scores define the ensemble weights.

In [ ]:
for name, pdir in VAL_BINARY_MODEL_DIRS.items():
    if not os.path.isdir(pdir):
        raise FileNotFoundError(
            f"Validation predictions not found: {pdir}. Run the "
            "validation-export cell of the corresponding training notebook "
            "(02, 05, 10 or 12) first.")

VALIDATION_DICE_SCORES = {}
print("Validation Dice of the base models:")
for name, pdir in VAL_BINARY_MODEL_DIRS.items():
    mean_dice, std_dice = evaluate_predictions_dir(pdir, VAL_GT_PATH)
    if mean_dice is None:
        raise RuntimeError(f"No matched validation predictions in {pdir}")
    VALIDATION_DICE_SCORES[name] = mean_dice
    print(f"   {name:20s}: {mean_dice:.4f} +/- {std_dice:.4f}")

# Top-2 scheme: the two models with the highest validation Dice, with weights
# proportional to their scores (zero weight for the remaining models).
_top2 = sorted(VALIDATION_DICE_SCORES, key=VALIDATION_DICE_SCORES.get)[-2:]
_top2_sum = sum(VALIDATION_DICE_SCORES[n] for n in _top2)
TOP2_WEIGHTS = [
    VALIDATION_DICE_SCORES[n] / _top2_sum if n in _top2 else 0.0
    for n in MODEL_NAMES
]

print("\nEnsemble weights:")
print("1. Simple average:")
for name in MODEL_NAMES:
    print(f"   {name:20s}: 0.2500")

dice_weights = np.array([VALIDATION_DICE_SCORES[name] for name in MODEL_NAMES])
dice_weights_norm = dice_weights / dice_weights.sum()
os.makedirs("results", exist_ok=True)
with open("results/ensemble_weights.json", "w") as f:
    json.dump({"validation_dice": VALIDATION_DICE_SCORES,
               "weights": {n: float(v) for n, v in zip(MODEL_NAMES, dice_weights_norm)}},
              f, indent=2)

print("2. Dice-weighted average:")
for name, weight in zip(MODEL_NAMES, dice_weights_norm):
    print(f"   {name:20s}: {weight:.4f}")

print("3. Top-2 (two best by validation Dice):")
for name, weight in zip(MODEL_NAMES, TOP2_WEIGHTS):
    if weight > 0:
        print(f"   {name:20s}: {weight:.4f}")

## 3b. Uniform validation Dice of all exported models (optional)

Any model that exports its binary validation predictions to `val_masks/<model>/`
is scored here with the same per-scene definition as above (and as the
test-set protocol): binarized predictions, nearest-neighbour upsampling to the
native ground-truth resolution, threshold 0.5, mean over scenes. This is the
single source of validation Dice for the paper; the soft Dice values logged
during training are early-stopping monitors and are not comparable across
frameworks.

In [ ]:
import glob

extra_dirs = sorted(d for d in glob.glob("val_masks/*") if os.path.isdir(d))
if extra_dirs:
    rows = []
    for d in extra_dirs:
        mean_dice, std_dice = evaluate_predictions_dir(d, VAL_GT_PATH)
        if mean_dice is None:
            print(f"   {os.path.basename(d):26s}: no matched predictions")
            continue
        rows.append((os.path.basename(d), mean_dice, std_dice))

    ens_tails = {os.path.basename(p) for p in VAL_BINARY_MODEL_DIRS.values()}
    print("Validation Dice, one definition for every exported model:")
    for name, m, s in sorted(rows, key=lambda r: -r[1]):
        tag = "  [ensemble member]" if name in ens_tails else ""
        print(f"   {name:26s} {m:.4f} +/- {s:.4f}{tag}")
else:
    print("val_masks/ is empty - export validation predictions first")

## 4. Averaging schemes

In [ ]:
def simple_average(masks):
    """Equal-weight pixel-wise mean of the mask stack."""
    return np.mean(np.stack(masks, axis=0), axis=0)


def weighted_average_by_dice(masks, model_names):
    """Weighted mean with weights proportional to validation Dice scores."""
    weights = np.array([VALIDATION_DICE_SCORES[name] for name in model_names])
    weights = weights / weights.sum()

    weighted = np.zeros_like(masks[0], dtype=np.float32)
    for i, mask in enumerate(masks):
        weighted += weights[i] * mask
    return weighted


def weighted_average_custom(masks, model_names, custom_weights):
    """Weighted mean with user-supplied weights (normalized to sum to 1)."""
    weights = np.array(custom_weights, dtype=np.float64)
    weights = weights / weights.sum()

    weighted = np.zeros_like(masks[0], dtype=np.float32)
    for i, mask in enumerate(masks):
        weighted += weights[i] * mask
    return weighted


def apply_averaging_methods(model_dirs, output_dir, mask_type):
    """Average per-image masks of all models and write the ensemble maps."""
    method_names = ["simple_average", "weighted_by_dice", "weighted_top2"]
    for method_name in method_names:
        (Path(output_dir) / method_name).mkdir(parents=True, exist_ok=True)

    model_names = list(model_dirs.keys())
    first_dir = Path(model_dirs[model_names[0]])
    pred_files = sorted([p.name for p in first_dir.glob("*_pred.png")])

    print(f"[{mask_type}] found {len(pred_files)} prediction files")

    written = 0
    for img_name in pred_files:
        masks = []
        shapes_ok = True

        for name in model_names:
            path = Path(model_dirs[name]) / img_name
            if not path.exists():
                shapes_ok = False
                break
            mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                shapes_ok = False
                break
            masks.append(mask.astype(np.float32))

        if not shapes_ok or len(masks) != len(model_names):
            continue

        # Bring all maps to a common size (that of the first model).
        target_shape = masks[0].shape
        for i in range(1, len(masks)):
            if masks[i].shape != target_shape:
                masks[i] = cv2.resize(masks[i], (target_shape[1], target_shape[0]),
                                      interpolation=cv2.INTER_LINEAR)

        avg_simple = simple_average(masks)
        cv2.imwrite(str(Path(output_dir) / "simple_average" / img_name),
                    avg_simple.astype(np.uint8))

        avg_weighted = weighted_average_by_dice(masks, model_names)
        cv2.imwrite(str(Path(output_dir) / "weighted_by_dice" / img_name),
                    avg_weighted.astype(np.uint8))

        avg_top2 = weighted_average_custom(masks, model_names, TOP2_WEIGHTS)
        cv2.imwrite(str(Path(output_dir) / "weighted_top2" / img_name),
                    avg_top2.astype(np.uint8))

        written += 1

    print(f"[{mask_type}] ensembles written for {written} images -> {output_dir}")

## 5. Scheme selection on the validation split

All six candidates (three schemes x two mask domains) are built from the
validation exports and scored against the validation ground truth; the
candidate with the highest validation Dice becomes the working configuration.

In [ ]:
val_results = {}

for domain, dirs in (("binary", VAL_BINARY_MODEL_DIRS),
                     ("probability", VAL_PROBABILITY_MODEL_DIRS)):
    out_dir = OUTPUT_DIRS[("val", domain)]
    apply_averaging_methods(dirs, out_dir, f"val / {domain}")

    for method_key in METHOD_TITLES:
        mean_dice, std_dice = evaluate_predictions_dir(
            Path(out_dir) / method_key, VAL_GT_PATH)
        if mean_dice is None:
            print(f"   {METHOD_TITLES[method_key]:26s} [{domain}]: no ensemble maps found")
            continue
        val_results[(domain, method_key)] = mean_dice

print("\n" + "=" * 70)
print("VALIDATION Dice of the candidate configurations")
print("=" * 70)
for (domain, method_key), mean_dice in sorted(val_results.items(),
                                              key=lambda kv: -kv[1]):
    print(f"   {METHOD_TITLES[method_key]:26s} [{domain:11s}]: {mean_dice:.4f}")

SELECTED_DOMAIN, SELECTED_METHOD = max(val_results, key=val_results.get)
print(f"\nSelected on validation: {METHOD_TITLES[SELECTED_METHOD]} "
      f"on {SELECTED_DOMAIN} masks "
      f"(validation Dice = {val_results[(SELECTED_DOMAIN, SELECTED_METHOD)]:.4f})")

## 6. Test-set evaluation

The configuration selected on the validation split is evaluated once on the
test split. Test scores of the base models and of the non-selected schemes are
printed for reference only.

In [ ]:
print("Base models (test):")
base_results = {}
for model_name, pred_dir in TEST_BINARY_MODEL_DIRS.items():
    mean_dice, std_dice = evaluate_predictions_dir(pred_dir, TEST_GT_PATH)
    if mean_dice is None:
        print(f"   {model_name:25s}: no matched predictions")
        continue
    base_results[model_name] = mean_dice
    print(f"   {model_name:25s}: {mean_dice:.4f} +/- {std_dice:.4f}")

test_results = {}
for domain, dirs in (("binary", TEST_BINARY_MODEL_DIRS),
                     ("probability", TEST_PROBABILITY_MODEL_DIRS)):
    out_dir = OUTPUT_DIRS[("test", domain)]
    apply_averaging_methods(dirs, out_dir, f"test / {domain}")

    for method_key in METHOD_TITLES:
        mean_dice, std_dice = evaluate_predictions_dir(
            Path(out_dir) / method_key, TEST_GT_PATH)
        if mean_dice is None:
            continue
        test_results[(domain, method_key)] = (mean_dice, std_dice)

sel_mean, sel_std = test_results[(SELECTED_DOMAIN, SELECTED_METHOD)]
print("\n" + "=" * 70)
print("ADOPTED CONFIGURATION (selected on the validation split)")
print("=" * 70)
print(f"{METHOD_TITLES[SELECTED_METHOD]} on {SELECTED_DOMAIN} masks: "
      f"test Dice = {sel_mean:.4f} +/- {sel_std:.4f}")

print("\nReference - test Dice of the non-selected candidates "
      "(not used for selection):")
for (domain, method_key), (mean_dice, std_dice) in sorted(
        test_results.items(), key=lambda kv: -kv[1][0]):
    if (domain, method_key) == (SELECTED_DOMAIN, SELECTED_METHOD):
        continue
    print(f"   {METHOD_TITLES[method_key]:26s} [{domain:11s}]: "
          f"{mean_dice:.4f} +/- {std_dice:.4f}")